In [ ]:
!pip install ultralytics


In [ ]:
pip install pillow

In [ ]:
!pip install opencv-python

In [ ]:
!pip install pyyaml

In [ ]:
import ultralytics
ultralytics.checks()

In [ ]:
import yaml


In [79]:
def write_yaml_file(data, file_path):
    """
    Write data to a YAML file.
    
    Args:
        data (dict): The data to write to the YAML file.
        file_path (str): The path to the YAML file.
    """
    if not isinstance(data, dict):
        raise ValueError("Data must be a dictionary.")
    
    if not isinstance(file_path, str):
        raise ValueError("File path must be a string.")
    
    # Ensure the directory exists
    import os
    directory = os.path.dirname(file_path)
    if not os.path.exists(directory):
        os.makedirs(directory)
    # Write the data to the YAML file
    with open(file_path, 'w') as file:
        yaml.dump(data, file, default_flow_style=False)

In [88]:
import os
from PIL import Image
import shutil

# YOLO formatında klasör yapısı oluştur
def prepare_yolo_structure():
    os.makedirs("datasets", exist_ok=True)
    os.makedirs("datasets/images/train", exist_ok=True)
    os.makedirs("datasets/labels/train", exist_ok=True)
    os.makedirs("datasets/images/val", exist_ok=True)
    os.makedirs("datasets/labels/val", exist_ok=True)

# YAML konfigürasyon dosyası oluştur
# data_yaml = {
#     'train': 'datasets/images/train',
#     'val': 'datasets/images/val',
#     'names': ['kanama', 'iskemi', 'normal'],
#     'nc': 3
# }



In [ ]:
# write_yaml_file(data_yaml, 'data.yaml')

In [ ]:
import numpy as np
import cv2

In [ ]:
from dicomutils import  dicomutils
utils = dicomutils()




In [ ]:
import pydicom
import tensorflow as tf


def load_dicom_data(folder_path, target_size=(256, 256)):
    """
    Load DICOM images from a folder and return as normalized numpy arrays
    """
    dicom_files = [f for f in os.listdir(folder_path) if f.endswith('.dcm')]
    images = []
    
    for file in dicom_files:
        try:
            # Load DICOM image
            ds = pydicom.dcmread(os.path.join(folder_path, file))
            img = ds.pixel_array
            
            # Normalize to 0-1 range
            if np.max(img) - np.min(img) > 0:  # Prevent division by zero
                img = (img - np.min(img)) / (np.max(img) - np.min(img))
            else:
                img = img.astype(float)
            
            # Resize if needed
            if img.shape != target_size:
                img = tf.image.resize(img[np.newaxis, ..., np.newaxis], target_size)
                img = img.numpy().squeeze()
            
            images.append(img)
        except Exception as e:
            print(f"Error processing {file}: {str(e)}")
            continue
    
    return np.array(images)


In [ ]:
# kanama_images = load_dicom_data("Kanama Veri Seti/DICOM")
# kanama_masks =  load_dicom_data("Kanama Veri Seti/Mask/kanama")


In [ ]:
# def convert_to_yolo_format(mask_path,class_id=0, img_size=256):
#     mask = np.array(Image.open(mask_path))
#     contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
#     yolo_lines = []
#     for cnt in contours:
#         # Filter out small contours
#         if cv2.contourArea(cnt) < 10:  # Adjust threshold as needed
#             continue

#         x,y,w,h = cv2.boundingRect(cnt)
#         # YOLO formatı: [class_id, x_center, y_center, width, height] (normalize edilmiş)
#         x_center = (x + w/2) / img_size
#         y_center = (y + h/2) / img_size
#         width = w / img_size
#         height = h / img_size
#         yolo_lines.append(f"{class_id} {x_center} {y_center} {width} {height}")
    
#     return "\n".join(yolo_lines)

In [ ]:
# def convert_to_yolo_format(masks, class_id=0, img_size=256):
#     """
#     Convert masks to YOLO format labels
#     """
#     yolo_labels = []
    
#     for mask in masks:
#         # Ensure mask is in uint8 format for findContours
#         mask_uint8 = (mask * 255).astype(np.uint8)
        
#         # Find contours in the mask
#         contours, _ = cv2.findContours(mask_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
#         label_lines = []
#         for cnt in contours:
#             # Filter out small contours
#             # if cv2.contourArea(cnt) < 10:  # Adjust threshold as needed
#             #     continue
                
#             # Get bounding box
#             x, y, w, h = cv2.boundingRect(cnt)
            
#             # Convert to YOLO format (normalized)
#             x_center = (x + w/2) / img_size
#             y_center = (y + h/2) / img_size
#             width = w / img_size
#             height = h / img_size
            
#             label_lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")
        
#         yolo_labels.append("\n".join(label_lines))
    
#     return yolo_labels

In [72]:
def convert_mask_to_yolo_seg(masks, class_id=0, img_size=256):
    """
    Convert binary masks to YOLO segmentation format (polygon points)
    Handles either a single mask or an array of masks
    """
    all_labels = []
    
    # Handle case where masks is a list/array of masks
    if isinstance(masks, np.ndarray) and masks.ndim > 2:
        # Process each mask individually
        for mask in masks:
            all_labels.append(process_single_mask(mask, class_id, img_size))
        return all_labels
    else:
        # Process a single mask
        return [process_single_mask(masks, class_id, img_size)]

def process_single_mask(mask, class_id=0, img_size=256):
    """Process a single binary mask to YOLO segmentation format"""
    # Convert mask to uint8 format for findContours
    mask_uint8 = (mask * 255).astype(np.uint8)
    
    # Ensure mask is single channel
    if mask_uint8.ndim > 2:
        # If mask has more than 2 dimensions, convert to grayscale
        mask_uint8 = cv2.cvtColor(mask_uint8, cv2.COLOR_RGB2GRAY)
    
    # Find contours in the mask
    contours, _ = cv2.findContours(mask_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    label_lines = []
    for cnt in contours:
        # Filter out small contours
        if cv2.contourArea(cnt) < 10:  # Adjust threshold as needed
            continue
        
        # Create polygon points
        points = []
        for point in cnt:
            x, y = point[0]
            # Normalize coordinates
            x_norm = x / img_size
            y_norm = y / img_size
            points.extend([x_norm, y_norm])
        
        # Format as YOLO segmentation line: class_id x1 y1 x2 y2 ...
        if points:  # Make sure points isn't empty
            points_str = " ".join([f"{p:.6f}" for p in points])
            label_lines.append(f"{class_id} {points_str}")
    
    return "\n".join(label_lines)

In [ ]:

import pydicom
import tensorflow as tf


def load_mask_data(dicom_folder, overlay_folder, target_size=(256, 256)):
    """
    Load both DICOM images and corresponding mask overlays
    """
    dicom_files = [f for f in os.listdir(dicom_folder) if f.endswith('.dcm')]
    images = []
    masks = []
    filenames = []  # Store filenames for reference
    
    for file in dicom_files:
        try:
            # Load DICOM image
            ds = pydicom.dcmread(os.path.join(dicom_folder, file))
            img = ds.pixel_array

            # Normalize to 0-1 range
            if np.max(img) - np.min(img) > 0:  # Prevent division by zero
                img = (img - np.min(img)) / (np.max(img) - np.min(img))
            else:
                img = img.astype(float)
            
            # Resize if needed
            if img.shape != target_size:
                img = tf.image.resize(img[np.newaxis, ..., np.newaxis], target_size)
                img = img.numpy().squeeze()
            
            # Load overlay mask
            overlay_path = os.path.join(overlay_folder, file.replace('.dcm', '.png'))
            print(f"Looking for mask at: {overlay_path}")
            
            if os.path.exists(overlay_path):
                mask = np.array(Image.open(overlay_path).convert('L'))
                mask = (mask > 0).astype(np.float32)  # Convert to binary mask
                
                # Resize mask if needed
                if mask.shape != target_size:
                    mask = tf.image.resize(mask[np.newaxis, ..., np.newaxis], target_size)
                    mask = mask.numpy().squeeze()
                
                images.append(img)
                masks.append(mask)
                filenames.append(file)
            else:
                print(f"Mask not found for {file}")
                
        except Exception as e:
            print(f"Error processing {file}: {str(e)}")
            continue
    
    return np.array(images), np.array(masks), filenames

In [76]:

# for i, (img, mask) in enumerate(zip(kanama_images, kanama_masks)):
#     img_pil = Image.fromarray((img*255).astype(np.uint8))
#     img_pil.save(f"datasets/images/train/kanama_{i}.png")
    
#     yolo_label = convert_mask_to_yolo(mask)
#     with open(f"datasets/labels/train/kanama_{i}.txt", "w") as f:
#         f.write(yolo_label)



# def process_dataset(dicom_folder, mask_folder, class_id, class_name, split_ratio=0.8):
#     """
#     Process a dataset for a specific class (kanama, iskemi, normal)
#     """
#     print(f"Processing {class_name} dataset...")
#     # Load the data
#     images, masks, filenames = load_mask_data(dicom_folder, mask_folder)
    
#     if len(images) == 0:
#         print(f"No valid image-mask pairs found for {class_name}")
#         return
        
#     print(f"Loaded {len(images)} image-mask pairs for {class_name}")
    
#     # Convert masks to YOLO format
#     yolo_labels = convert_mask_to_yolo_seg(masks, class_id)
    
#     # Split into train/val sets
#     split_idx = int(len(images) * split_ratio)
#     train_indices = list(range(split_idx))
#     val_indices = list(range(split_idx, len(images)))
    
#     # Save images and labels
#     for idx, set_type in [(train_indices, "train"), (val_indices, "val")]:
#         for i, index in enumerate(idx):
#             if index >= len(images):
#                 continue
                
#             # Convert image to uint8 for saving
#             img = images[index]
#             img_uint8 = (img * 255).astype(np.uint8)
            
#             # Handle grayscale images properly
#             if len(img_uint8.shape) == 2:
#                 img_pil = Image.fromarray(img_uint8, mode='L')
#             else:
#                 img_pil = Image.fromarray(img_uint8)
                
#             # Save image
#             img_filename = f"{class_name}_{i}.png"
#             img_path = f"datasets/images/{set_type}/{img_filename}"
#             img_pil.save(img_path)
            
#             # Save label
#             label_path = f"datasets/labels/{set_type}/{class_name}_{i}.txt"
#             with open(label_path, "w") as f:
#                 f.write(yolo_labels[index])
            
#             print(f"Saved {img_path} and corresponding label")


def process_dataset(dicom_folder, mask_folder, class_id, class_name, split_ratio=0.8):
    """
    Process a dataset for a specific class (kanama, iskemi, normal)
    """
    print(f"Processing {class_name} dataset...")
    # Load the data
    images, masks, filenames = load_mask_data(dicom_folder, mask_folder)
    
    if len(images) == 0:
        print(f"No valid image-mask pairs found for {class_name}")
        return
        
    print(f"Loaded {len(images)} image-mask pairs for {class_name}")
    
    # Debug information
    print(f"Mask array shape: {masks.shape}")
    print(f"Mask data type: {masks.dtype}")
    
    # Convert masks to YOLO format
    yolo_labels = convert_mask_to_yolo_seg(masks, class_id)
    
    # Split into train/val sets
    split_idx = int(len(images) * split_ratio)
    train_indices = list(range(split_idx))
    val_indices = list(range(split_idx, len(images)))
    
    # Save images and labels
    for set_name, indices in [("train", train_indices), ("val", val_indices)]:
        for i, index in enumerate(indices):
            if index >= len(images):
                continue
                
            # Convert image to uint8 for saving
            img = images[index]
            img_uint8 = (img * 255).astype(np.uint8)
            
            # Handle grayscale images properly
            if len(img_uint8.shape) == 2:
                img_pil = Image.fromarray(img_uint8, mode='L')
            else:
                img_pil = Image.fromarray(img_uint8)
                
            # Save image
            img_filename = f"{class_name}_{i}.png"
            img_path = f"datasets/images/{set_name}/{img_filename}"
            img_pil.save(img_path)
            
            # Save label
            label_path = f"datasets/labels/{set_name}/{class_name}_{i}.txt"
            with open(label_path, "w") as f:
                f.write(yolo_labels[index])
            
            print(f"Saved {img_path} and corresponding label")

In [94]:
 # Define class mapping
classes = {
        0: {"name": "kanama", "dicom": "Kanama Veri Seti/DICOM", "mask": "Kanama Veri Seti/Mask/kanama"},
        1: {"name": "iskemi", "dicom": "Iskemi Veri Seti/DICOM", "mask": "Iskemi Veri Seti/Mask/iskemi"},
        # 2: {"name": "normal", "dicom": "Normal Veri Seti/DICOM", "mask": "Normal Veri Seti/Mask/normal"}
}

In [89]:
# Örnek veri dönüşümü (Kanama veri seti için)
prepare_yolo_structure()

In [95]:

# Process each class
for class_id, class_info in classes.items():
    if os.path.exists(class_info["dicom"]) and os.path.exists(class_info["mask"]):
        process_dataset(
            class_info["dicom"], 
            class_info["mask"], 
            class_id, 
            class_info["name"]
        )
    else:
        print(f"Directories not found for {class_info['name']}")



Processing kanama dataset...
Looking for mask at: Kanama Veri Seti/Mask/kanama\10002.png
Looking for mask at: Kanama Veri Seti/Mask/kanama\10033.png
Looking for mask at: Kanama Veri Seti/Mask/kanama\10036.png
Looking for mask at: Kanama Veri Seti/Mask/kanama\10039.png
Looking for mask at: Kanama Veri Seti/Mask/kanama\10045.png
Looking for mask at: Kanama Veri Seti/Mask/kanama\10046.png
Looking for mask at: Kanama Veri Seti/Mask/kanama\10047.png
Looking for mask at: Kanama Veri Seti/Mask/kanama\10049.png
Looking for mask at: Kanama Veri Seti/Mask/kanama\10050.png
Looking for mask at: Kanama Veri Seti/Mask/kanama\10052.png
Looking for mask at: Kanama Veri Seti/Mask/kanama\10063.png
Looking for mask at: Kanama Veri Seti/Mask/kanama\10067.png
Looking for mask at: Kanama Veri Seti/Mask/kanama\10068.png
Looking for mask at: Kanama Veri Seti/Mask/kanama\10075.png
Looking for mask at: Kanama Veri Seti/Mask/kanama\10078.png
Looking for mask at: Kanama Veri Seti/Mask/kanama\10094.png
Looking for

In [80]:
# Create YAML configuration file
data_yaml = {
    'train': 'images/train',
    'val': 'images/val',
    'names': [class_info["name"] for _, class_info in classes.items()],
    'nc': len(classes),
    'task': 'segment'
}


In [82]:

# write_yaml_file(data_yaml, 'data.yaml')
with open( 'data.yaml', 'w') as file:
    yaml.dump(data_yaml, file, default_flow_style=False)

print("YAML configuration file created.")

YAML configuration file created.


In [96]:
import os
import glob

def check_yolo_dataset(dataset_dir):
    """Check for empty label files and mismatches between images and labels"""
    for split in ['train', 'val']:
        images_dir = os.path.join(dataset_dir, 'images', split)
        labels_dir = os.path.join(dataset_dir, 'labels', split)
        
        image_files = set([os.path.splitext(os.path.basename(f))[0] 
                          for f in glob.glob(f"{images_dir}/*.png") + glob.glob(f"{images_dir}/*.jpg")])
        label_files = set([os.path.splitext(os.path.basename(f))[0] 
                          for f in glob.glob(f"{labels_dir}/*.txt")])
        
        # Check for images without labels
        img_no_label = image_files - label_files
        if img_no_label:
            print(f"WARNING: {len(img_no_label)} images have no labels in {split} set")
            print(f"Examples: {list(img_no_label)[:5]}")
        
        # Check for labels without images
        label_no_img = label_files - image_files
        if label_no_img:
            print(f"WARNING: {len(label_no_img)} labels have no images in {split} set")
            print(f"Examples: {list(label_no_img)[:5]}")
        
        # Check for empty label files
        empty_labels = []
        for label in label_files:
            label_path = os.path.join(labels_dir, f"{label}.txt")
            if os.path.exists(label_path) and os.path.getsize(label_path) == 0:
                empty_labels.append(label)
        
        if empty_labels:
            print(f"WARNING: {len(empty_labels)} empty label files in {split} set")
            print(f"Examples: {empty_labels[:5]}")



In [97]:
# Call this function with your dataset directory
check_yolo_dataset("datasets")

Examples: ['kanama_814', 'kanama_433']
Examples: ['kanama_77']


In [98]:
from ultralytics import YOLO

# Model seçimi (Segmentasyon için YOLOv8x-seg)
model = YOLO('yolov8x-seg.pt')  # Önceden eğitilmiş model

# Eğitim parametreleri
train_args = {
    'data': 'data.yaml',
    'epochs': 50,
    'batch': 8,
    'imgsz': 256,
    'device': '0' if tf.config.list_physical_devices('GPU') else 'cpu',
    'optimizer': 'Adam',
    'lr0': 0.001,
    'name': 'stroke_detection'
}

# Eğitimi başlat
results = model.train(**train_args)

Ultralytics 8.3.107  Python-3.8.10 torch-2.4.1+cpu CPU (13th Gen Intel Core(TM) i7-1355U)
engine\trainer: task=segment, mode=train, model=yolov8x-seg.pt, data=data.yaml, epochs=50, time=None, patience=100, batch=8, imgsz=256, save=True, save_period=-1, cache=False, device=cpu, workers=8, project=None, name=stroke_detection, exist_ok=False, pretrained=True, optimizer=Adam, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_crop=False, show_labels=True, show_conf=True, show_boxes=True, line_wi

train: Scanning F:\D\dataset\saglik-acikveri-teknofest-2021\datasets\labels\train... 1766 images, 2 backgrounds, 0 corrupt: 100%|██████████| 1766/1766 [00:01<00:00, 1561.60it/s]


train: New cache created: F:\D\dataset\saglik-acikveri-teknofest-2021\datasets\labels\train.cache


val: Scanning F:\D\dataset\saglik-acikveri-teknofest-2021\datasets\labels\val... 443 images, 1 backgrounds, 0 corrupt: 100%|██████████| 443/443 [00:00<00:00, 1565.37it/s]


val: New cache created: F:\D\dataset\saglik-acikveri-teknofest-2021\datasets\labels\val.cache
Plotting labels to runs\segment\stroke_detection\labels.jpg... 
optimizer: Adam(lr=0.001, momentum=0.937) with parameter groups 106 weight(decay=0.0), 117 weight(decay=0.0005), 116 bias(decay=0.0)
TensorBoard: model graph visualization added 
Image sizes 256 train, 256 val
Using 0 dataloader workers
Logging results to runs\segment\stroke_detection
Starting training for 50 epochs...

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss  Instances       Size


       1/50         0G      2.852      4.697      3.289      1.994         19        256:  14%|█▍        | 31/221 [03:50<23:33,  7.44s/it]


KeyboardInterrupt: 

In [ ]:
# Validation seti üzerinde değerlendirme
metrics = model.val()
print(f"mAP50-95: {metrics.box.map:.4f}")

# Örnek tahmin
sample_img = "datasets/images/val/iskemi_0.png"
results = model.predict(sample_img, save=True, imgsz=256)

# Sonuçları görselleştirme
for result in results:
    result.show()
    result.save("prediction.jpg")

In [ ]:
test_dir = "YarısmaVeriSeti_1.Oturum/DICOM"
os.makedirs("YarısmaVeriSeti_1.Oturum/MASKS_YOLO", exist_ok=True)

for img_file in os.listdir(test_dir):
    if img_file.endswith('.dcm'):
        img_path = os.path.join(test_dir, img_file)
        img = pydicom.dcmread(img_path).pixel_array
        img = (img - np.min(img)) / (np.max(img) - np.min(img))
        img_pil = Image.fromarray((img*255).astype(np.uint8))
        img_pil.save("temp.png")
        
        results = model.predict("temp.png", imgsz=256)
        
        # Sonuçları kaydet
        for result in results:
            result.save_txt(f"YarısmaVeriSeti_1.Oturum/MASKS_YOLO/{img_file[:-4]}.txt")
            result.save(f"YarısmaVeriSeti_1.Oturum/MASKS_YOLO/{img_file[:-4]}_vis.png")